# Aaliyah John-Harry
# DATA 612 Project 1: Global Baseline Predictors and RMSE
## 06/04/2026

## Business Context

This system recommends movies to users on a streaming platform. The goal is to estimate how a user may rate a movie based on prior ratings. These predictions could help rank movies for a user's home page, suggest titles they are likely to enjoy, or provide a basic fallback recommendation model before using more advanced methods.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## Dataset

I created a small movie-rating dataset with 7 users and 6 movies.

Each row represents one observed rating, ranging from 1 to 5 stars. Missing ratings are not listed in the raw dataframe, but appear as blank values after pivoting into a user-item matrix.

In [2]:
ratings_data = [
    # user, movie, rating
    ('Margot R', 'The Drama', 5),
    ('Margot R', 'Scream 7', 4),
    ('Margot R', 'Shelter', 5),
    ('Margot R', 'You, Me & Tuscany', 4),
    ('Margot R', 'Sinners', 3),

    ('Zendaya C', 'The Drama', 4),
    ('Zendaya C', 'Scream 7', 5),
    ('Zendaya C', 'Now You See Me', 4),
    ('Zendaya C', 'You, Me & Tuscany', 3),

    ('Thomas H', 'Scream 7', 2),
    ('Thomas H', 'Shelter', 3),
    ('Thomas H', 'Now You See Me', 4),
    ('Thomas H', 'Sinners', 2),

    ('Kofi S', 'The Drama', 2),
    ('Kofi S', 'Shelter', 2),
    ('Kofi S', 'Now You See Me', 3),
    ('Kofi S', 'You, Me & Tuscany', 2),
    ('Kofi S', 'Sinners', 1),

    ('Ylan N', 'The Drama', 5),
    ('Ylan N', 'Scream 7', 4),
    ('Ylan N', 'Now You See Me', 5),
    ('Ylan N', 'Sinners', 4),

    ('Gianni P', 'Scream 7', 3),
    ('Gianni P', 'Shelter', 4),
    ('Gianni P', 'You, Me & Tuscany', 5),
    ('Gianni P', 'Sinners', 4),

    ('Alix L', 'Scream 7', 3),
    ('Alix L', 'Shelter', 4),
    ('Alix L', 'You, Me & Tuscany', 5),
    ('Alix L', 'Sinners', 4),
]

ratings = pd.DataFrame(ratings_data, columns=['user_id', 'item_id', 'rating'])
ratings

,user_id,item_id,rating
0,Margot R,The Drama,5
1,Margot R,Scream 7,4
2,Margot R,Shelter,5
3,Margot R,"You, Me & Tuscany",4
4,Margot R,Sinners,3
5,Zendaya C,The Drama,4
6,Zendaya C,Scream 7,5
7,Zendaya C,Now You See Me,4
8,Zendaya C,"You, Me & Tuscany",3
9,Thomas H,Scream 7,2


## User-Item Matrix

- I converted the dataset into a user-item matrix, with the users as rows and movies as columns.
- Each cell contains the rating a user gave to a movie.
    - Missing values (`NaN`) represent movies that a particular user has not rated. 


In [4]:
user_item_matrix = ratings.pivot(index='user_id', columns='item_id', values='rating')
user_item_matrix

item_id,Now You See Me,Scream 7,Shelter,Sinners,The Drama,"You, Me & Tuscany"
user_id,,,,,,
Alix L,NaN,3.0,4.0,4.0,NaN,5.0
Gianni P,NaN,3.0,4.0,4.0,NaN,5.0
Kofi S,3.0,NaN,2.0,1.0,2.0,2.0
Margot R,NaN,4.0,5.0,3.0,5.0,4.0
Thomas H,4.0,2.0,3.0,2.0,NaN,NaN
Ylan N,5.0,4.0,NaN,4.0,5.0,NaN
Zendaya C,4.0,5.0,NaN,NaN,4.0,3.0


## Training and Test Data

- I split the dataset into training and test sets, with 80% of the ratings assigned to the training set and 20% to the test set.
- I will use the training set to build the recommendation model, while the test set will be used to evaluate how accurately the model predicts ratings it has not seen before. 

In [5]:
train_df, test_df = train_test_split(
    ratings,
    test_size=0.2,
    random_state=27,
    stratify=None
)

print(f"Total ratings: {len(ratings)}")
print(f"Training ratings: {len(train_df)}")
print(f"Test ratings: {len(test_df)}")

display(train_df.head())
display(test_df.head())

Total ratings: 30
Training ratings: 24
Test ratings: 6


,user_id,item_id,rating
9,Thomas H,Scream 7,2
25,Gianni P,Sinners,4
12,Thomas H,Sinners,2
26,Alix L,Scream 7,3
0,Margot R,The Drama,5


,user_id,item_id,rating
29,Alix L,Sinners,4
15,Kofi S,Now You See Me,3
11,Thomas H,Now You See Me,4
7,Zendaya C,Now You See Me,4
21,Ylan N,Sinners,4


## Raw Mean Rating

- The average rating in the training dataset was approximately **~3.542**.
- This will be the baseline prediction and provide a simple starting point for evaluating the recommender system before I incorporate the user and item biases.

In [19]:
global_mean = train_df['rating'].mean()
print(f"Mean rating from training data: {global_mean:.3f}")

train_df = train_df.copy()
test_df = test_df.copy()

train_df['pred_global_mean'] = global_mean
test_df['pred_global_mean'] = global_mean

train_df.head(10)

Mean rating from training data: 3.542


,user_id,item_id,rating,pred_global_mean,pred_baseline
9,Thomas H,Scream 7,2,3.541667,2.631672
25,Gianni P,Sinners,4,3.541667,3.343386
12,Thomas H,Sinners,2,3.541667,2.312831
26,Alix L,Scream 7,3,3.541667,3.585838
0,Margot R,The Drama,5,3.541667,4.279828
22,Gianni P,Scream 7,3,3.541667,3.662227
23,Gianni P,Shelter,4,3.541667,4.010053
3,Margot R,"You, Me & Tuscany",4,3.541667,4.184212
6,Zendaya C,Scream 7,5,3.541667,3.631672
18,Ylan N,The Drama,5,3.541667,4.484590


## RMSE and MAE

- To evaluate the performance of the baseline model, I calculated both RMSE and MAE for the training and test datasets. 
    - They measure the difference between the actual ratings and the predicted ratings from the global average model.
- The results provide a benchmark for model performance and I will use them to compare whether incorporating user and item biases improves prediction accuracy.

In [20]:
def evaluate_predictions(actual, predicted):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    return rmse, mae

train_global_rmse, train_global_mae = evaluate_predictions(train_df['rating'], train_df['pred_global_mean'])
test_global_rmse, test_global_mae = evaluate_predictions(test_df['rating'], test_df['pred_global_mean'])

print("Average Model")
print(f"Training RMSE: {train_global_rmse:.3f}")
print(f"Training MAE:  {train_global_mae:.3f}")
print(f"Test RMSE:     {test_global_rmse:.3f}")
print(f"Test MAE:      {test_global_mae:.3f}")

Average Model
Training RMSE: 1.224
Training MAE:  1.080
Test RMSE:     0.473
Test MAE:      0.472


## User Biases

I calculated user biases to account for differences in rating tendencies among users. Some users consistently rate movies higher than average, while others are more conservative in their ratings. I also regularized it to prevent users with fewer ratings from having an outsized impact on the bias estimates.

Users with positive biases generally rate movies more favorably, while users with negative biases tend to be more critical.

For example, **Ylan N** has the highest positive bias (+0.675), suggesting he typically rates movies about 0.68 points above average. In contrast, **Kofi S** has the largest negative bias (-1.194), indicating that his ratings are usually lower than the average user. 

In [11]:
reg_user = 2

user_stats = train_df.groupby('user_id')['rating'].agg(['mean', 'count']).reset_index()
user_stats['user_bias'] = (user_stats['mean'] - global_mean) * (user_stats['count'] / (user_stats['count'] + reg_user))
user_biases = dict(zip(user_stats['user_id'], user_stats['user_bias']))

user_stats

,user_id,mean,count,user_bias
0,Alix L,4.000000,2,0.229167
1,Gianni P,4.000000,4,0.305556
2,Kofi S,1.750000,4,-1.194444
3,Margot R,4.200000,5,0.470238
4,Thomas H,2.333333,3,-0.725000
5,Ylan N,4.666667,3,0.675000
6,Zendaya C,4.000000,3,0.275000


## Item Biases

I calculated item biases to capture the differences in how movies are rated on average. To isolate the effect of each movie, I first took user biases into account before calculating the item bias. I also regularized here to prevent movies with fewer ratings from having an outsized influence on the results.

The item bias really just shows whether a movie tends to receive ratings above or below what would be expected after accounting for the global average rating and individual user rating tendencies.

- Positive item biases indicate movies that are generally rated more favorably than expected.
- Negative item biases indicate movies that tend to receive lower ratings than expected.

So in this case, **The Drama** has an item bias of approximately **+0.268**, suggesting that users tend to rate it about 0.27 points higher than expected after accounting for user biases. Similarly, **You, Me & Tuscany** (+0.172) and **Now You See Me** (+0.261) also receive slightly higher ratings than expected.

On the other hand, **Scream 7** has an item bias of approximately **-0.185**, indicating that it tends to receive ratings slightly below expectation. **Sinners** has the largest negative bias (-0.504), suggesting that it is generally rated about half a point lower than expected across users.

In [12]:
reg_item = 2

item_biases = {}
item_rows = []

for item, group in train_df.groupby('item_id'):
    adjusted_ratings = []
    for _, row in group.iterrows():
        user = row['user_id']
        rating = row['rating']
        bu = user_biases.get(user, 0)
        adjusted_ratings.append(rating - global_mean - bu)
    
    n = len(adjusted_ratings)
    item_bias = sum(adjusted_ratings) / (n + reg_item)
    item_biases[item] = item_bias
    item_rows.append({
        'item_id': item,
        'count': n,
        'regularized_item_bias': item_bias
    })

item_stats = pd.DataFrame(item_rows)
item_stats

,item_id,count,regularized_item_bias
0,Now You See Me,1,0.261111
1,Scream 7,6,-0.184995
2,Shelter,4,0.162831
3,Sinners,4,-0.503836
4,The Drama,4,0.267923
5,"You, Me & Tuscany",5,0.172307


## Baseline Prediction

- I generated the baseline predictions by combining the global average rating with the corresponding user and item biases.
- This gave me a more personalized estimate than the raw average model by accounting for differences in user preferences and movie ratings. 

In [14]:
def baseline_predict(user_id, item_id, global_mean, user_biases, item_biases, min_rating=1, max_rating=5):
    bu = user_biases.get(user_id, 0)
    bi = item_biases.get(item_id, 0)
    pred = global_mean + bu + bi
    return np.clip(pred, min_rating, max_rating)

train_df['pred_baseline'] = train_df.apply(
    lambda row: baseline_predict(row['user_id'], row['item_id'], global_mean, user_biases, item_biases),
    axis=1
)

test_df['pred_baseline'] = test_df.apply(
    lambda row: baseline_predict(row['user_id'], row['item_id'], global_mean, user_biases, item_biases),
    axis=1
)

train_df.head(10)

,user_id,item_id,rating,pred_global_mean,pred_baseline
9,Thomas H,Scream 7,2,3.541667,2.631672
25,Gianni P,Sinners,4,3.541667,3.343386
12,Thomas H,Sinners,2,3.541667,2.312831
26,Alix L,Scream 7,3,3.541667,3.585838
0,Margot R,The Drama,5,3.541667,4.279828
22,Gianni P,Scream 7,3,3.541667,3.662227
23,Gianni P,Shelter,4,3.541667,4.010053
3,Margot R,"You, Me & Tuscany",4,3.541667,4.184212
6,Zendaya C,Scream 7,5,3.541667,3.631672
18,Ylan N,The Drama,5,3.541667,4.484590


## Evaluate the Baseline Predictor

- I evaluated the baseline predictor using RMSE and MAE on both the training and test datasets.
- The error values were noticeably lower than those of the raw average model, showing that incorporating user and item biases improved the quality of the predictions.

By accounting for differences in user rating behavior and movie popularity, the baseline model was better able to capture patterns in the data and provide more accurate rating estimates.

In [15]:
train_base_rmse, train_base_mae = evaluate_predictions(train_df['rating'], train_df['pred_baseline'])
test_base_rmse, test_base_mae = evaluate_predictions(test_df['rating'], test_df['pred_baseline'])

print("Baseline Bias Model")
print(f"Training RMSE: {train_base_rmse:.3f}")
print(f"Training MAE:  {train_base_mae:.3f}")
print(f"Test RMSE:     {test_base_rmse:.3f}")
print(f"Test MAE:      {test_base_mae:.3f}")

Baseline Bias Model
Training RMSE: 0.656
Training MAE:  0.555
Test RMSE:     0.522
Test MAE:      0.413


## Summary of Results

While the baseline model achieved lower MAE on the test set, its test RMSE was slightly higher than the global average model. Given the small size of the dataset, these differences are likely influenced by the limited number of observations in the test set.

In [16]:
results = pd.DataFrame({
    'Model': ['Global Average', 'Baseline with User + Item Bias'],
    'Train RMSE': [train_global_rmse, train_base_rmse],
    'Train MAE': [train_global_mae, train_base_mae],
    'Test RMSE': [test_global_rmse, test_base_rmse],
    'Test MAE': [test_global_mae, test_base_mae]
})

results

,Model,Train RMSE,Train MAE,Test RMSE,Test MAE
0,Global Average,1.224036,1.079861,0.473242,0.472222
1,Baseline with User + Item Bias,0.655966,0.555299,0.521873,0.413029


## Test Predictions

The table below compares the actual ratings from the test set with the ratings predicted by the two models. The error columns show how far each prediction was from the true rating (smaller values mean more accurate predictions).

The test predictions show that the baseline model often gave estimates that were closer to the actual ratings than the global average model. In 4 of the 6 test cases, the baseline model resulted in a smaller prediction error. However, the baseline model also produced a few larger errors, which contributed to its slightly higher RMSE. 

These results highlight how incorporating user and item biases can improve many individual predictions while still being sensitive to occasional larger prediction errors.

In [17]:
test_predictions = test_df[['user_id', 'item_id', 'rating', 'pred_global_mean', 'pred_baseline']].copy()
test_predictions['global_error'] = test_predictions['rating'] - test_predictions['pred_global_mean']
test_predictions['baseline_error'] = test_predictions['rating'] - test_predictions['pred_baseline']
test_predictions

,user_id,item_id,rating,pred_global_mean,pred_baseline,global_error,baseline_error
29,Alix L,Sinners,4,3.541667,3.266997,0.458333,0.733003
15,Kofi S,Now You See Me,3,3.541667,2.608333,-0.541667,0.391667
11,Thomas H,Now You See Me,4,3.541667,3.077778,0.458333,0.922222
7,Zendaya C,Now You See Me,4,3.541667,4.077778,0.458333,-0.077778
21,Ylan N,Sinners,4,3.541667,3.712831,0.458333,0.287169
27,Alix L,Shelter,4,3.541667,3.933664,0.458333,0.066336


## Generate Recommendations for One User

To demonstrate how the recommender system can be used in practice, I predicted the ratings for movies that one user has not rated yet and ranked them from highest to lowest predicted rating.

For **Thomas H**, the model predicts that **The Drama** would be the most preferred unrated movie, followed by **You, Me & Tuscany**. These recommendations are based on the global average rating, user bias, and item bias calculated earlier.

In [18]:
target_user = 'Thomas H'

all_items = ratings['item_id'].unique()
rated_items = ratings.loc[ratings['user_id'] == target_user, 'item_id'].unique()
unrated_items = [item for item in all_items if item not in rated_items]

recommendations = pd.DataFrame({
    'user_id': target_user,
    'item_id': unrated_items
})
recommendations['predicted_rating'] = recommendations['item_id'].apply(
    lambda item: baseline_predict(target_user, item, global_mean, user_biases, item_biases)
)

recommendations.sort_values('predicted_rating', ascending=False)

,user_id,item_id,predicted_rating
0,Thomas H,The Drama,3.084590
1,Thomas H,"You, Me & Tuscany",2.988974


## Conclusion

In this project, I built a simple movie recommender system using global baseline predictors and evaluated its performance using RMSE and MAE. Starting with the global average rating provided a basic benchmark for prediction, while incorporating user and item biases allowed the model to account for differences in individual rating behavior and movie popularity.

The results showed that user and item biases can improve prediction quality by providing more personalized estimates than the overall average alone. Although the dataset used in this project was relatively small, the approach demonstrated the key concepts behind collaborative filtering and recommendation systems.